# Results — Rydberg-informed SAT code discovery

**13 November 2026 · Intermediate numerical research results**

**Question:** When the physical error probabilities are known, what changes when we require a stabilizer to be a symmetry of the Rydberg Hamiltonian?

This is a **complete, exhaustively checked four-physical-qubit \([[4,1]]\) search**, not a claim of scalable full-stack QEC. We use **all 16 Pauli weights** from the two-atom Rydberg noise channel. To obtain a four-qubit test, we apply that same error channel **independently to atoms (1,2) and (3,4)**. The resulting distribution contains **256** Pauli strings.

We compare unseeded code discovery with Hamiltonian-seeded code discovery, both with and without mandatory correction of every single-qubit \(Z\) error. These modes have different guarantees; within each mode the seed comparisons are matched. 

## 1. Where the error probabilities come from

Our earlier calculation used a **reduced three-level neutral-atom Hamiltonian**, three Rydberg decay branches, corrected branching for intermediate-state scattering, and assumed Gaussian common detuning fluctuations. It produced a four-dimensional computational-input to 16-dimensional physical-output channel, and then a Pauli-twirled distribution on the computational output.

A two-atom error is represented as \(P\in\{I,X,Y,Z\}^{\otimes2}\), with weight \(q_P\). Its 16 weights add to **0.9984648623**, not 1: the omitted weight is residual Rydberg occupation or entry into effective loss states. We **do not renormalize this away**.

For four atoms, **assuming two independent identical simultaneous pair operations**, 

\[
q_{P_1P_2P_3P_4}=q_{P_1P_2}\,q_{P_3P_4}.
\]

This gives \(P_{\mathrm{comp}}=(\sum_Pq_P)^2=0.9969320812\). About **0.3068%** of the input-averaged output is *outside* the computational states; no erasure/loss decoder is included in the present SAT study.

The two-pair assumption is a model extension, **not** a measured four-atom channel. The original coherent and leakage information is not retained by this projected Pauli approximation.

In [ ]:
from pathlib import Path
import json, numpy as np, matplotlib.pyplot as plt
# Works from either the repository root or its Results directory.
base = Path.cwd() if (Path.cwd()/"SAT_Rydberg/results.json").exists() else Path.cwd()/"Results"
results = json.loads((base/"SAT_Rydberg/results.json").read_text())
pair_weights = json.loads((base/"rydberg_pair_pauli_weights.json").read_text())
p_comp = sum(pair_weights.values())**2
print(f"Pauli survival for two pairs: {100*p_comp:.8f}%")
print(f"Outside computational space: {100*(1-p_comp):.8f}%")
assert abs(p_comp-results["setup"]["two_pair_computational_survival"])<1e-12

## 2. What is the Hamiltonian seed?

For one qutrit define

\[
Q=|0\rangle\langle0|-|1\rangle\langle1|-|r\rangle\langle r|
 =\operatorname{diag}(1,-1,-1).
\]

The Rydberg-only Hamiltonian drives \(|1\rangle\leftrightarrow |r\rangle\) and contains diagonal detuning and blockade terms. Because \(Q\) gives the **same eigenvalue to** \(|1\rangle\) **and** \(|r\rangle\), it commutes with this Hamiltonian (provided the hyperfine \(|0\rangle\leftrightarrow|1\rangle\) drive is off). The pair operator \(Q_1Q_2\) therefore also commutes; on the two computational qubits it acts as \(ZZ\).

For our two-pair model we can require the stabilizer \(ZZII\), or both \(ZZII\) and \(IIZZ\). These are **genuine coherent Hamiltonian symmetries during the Rydberg-only pulse**, not necessarily symmetries of the noisy evolution. The *additional* discovered stabilizers are not guaranteed to commute with the Hamiltonian. This is why we call the test **Hamiltonian-seeded code search** rather than claiming that the whole discovered code is invariant under \(H(t)\).

In [ ]:
Q = np.diag([1., -1., -1.]); I = np.eye(3)
rydberg_drive = np.array([[0,0,0],[0,0,1],[0,1,0]],float)
ry_pop = np.diag([0,0,1]); H = np.kron(rydberg_drive,I) + np.kron(I,rydberg_drive) + 12*np.kron(ry_pop,ry_pop)
print("Hamiltonian commutator with Q⊗Q:",np.linalg.norm(np.kron(Q,Q)@H-H@np.kron(Q,Q)))
assert np.linalg.norm(np.kron(Q,Q)@H-H@np.kron(Q,Q))<1e-12

## 3. Complete finite-size stabilizer constraints

We search for \([[n,k,d]]=[[4,1,d]]\) stabilizer codes with **three commuting, independent general Pauli generators** \(g_1,g_2,g_3\), with entries from \(\{I,X,Y,Z\}\). A binary symplectic representation \(g=(x|z)\in\mathbb F_2^{8}\) turns commutation into Boolean XOR constraints:

\[
\langle g_i,g_j\rangle_s=x_i\cdot z_j+z_i\cdot x_j\pmod2=0.
\]

We require a true code of **distance at least two**:

\[
d=\min_{P\in N(S)\setminus S}\operatorname{wt}(P)\ge2.
\]

In the strict **\(\varepsilon\)-code** mode, require correction of \(\mathcal A=\{IIII,ZIII,IZII,IIZI,IIIZ\}\). For *every* pair of errors \(E_a,E_b\in\mathcal A\):

\[
E_a^\dagger E_b\in S \quad\text{or}\quad
\exists g\in S:\{g,E_a^\dagger E_b\}=0.
\]

That is the full **Pauli Knill–Laflamme correction condition**, including the possibility that two errors act identically on the code. This is stronger than simply detecting each error separately.

**Verification:** Z3 solves the Boolean formula. Independently, our Python code enumerates **all 11,475** possible rank-3 commuting stabilizer groups on four qubits and checks correction, distance, and seed constraints. This certifies the optimum for this specific \([[4,1]]\) instance.

## 4. Probability-weighted selection

**The probability weights do not replace the SAT constraints.** First we generate valid stabilizer codes; then we choose the one with the **highest exact probability of successful correction** under the entire 256-error distribution.

Each stabilizer syndrome \(s\) can correspond to four different logical effects. Errors differing by a stabilizer have the same effect and their probabilities are **added together**. For each syndrome, the decoder chooses the most probable class:

\[
p_{\mathrm{success}}=\sum_s \max_{\ell}\sum_{E\in(s,\ell)}q_E,
\qquad
p_{\mathrm{logical\ error}\mid\mathrm{comp}}
=\frac{p_{\mathrm{comp}}-p_{\mathrm{success}}}{p_{\mathrm{comp}}}.
\]

This is **exact maximum-likelihood Pauli decoding**, not a Monte Carlo estimate or a heuristic sum of detected errors. There is no probability estimate for correcting the omitted leakage states in this calculation.

In [ ]:
conditions = [("unseeded","No seed"), ("seed_ZZII","Seed ZZII"), ("seed_ZZII_IIZZ","Both seeds")]
print("STRICT ε-CODE: every single-Z error correctable, d≥2")
for key,label in conditions:
    record=results[key]
    best=record["best"]
    pct = f"{best['logical_failure_conditional_on_computational_survival']*100:.9f}%" if best else 'UNSAT'
    print(f"{label:12s} | feasible {record['hard_kl_feasible']:4d} | SAT={record['smt_status']:5s} | error {pct}")
    if best: print("  stabilizers:", ", ".join(best["generators"]), "; distance:", best["distance"])

### Result A — strict \(\varepsilon\)-codes (all single-qubit \(Z\) errors correctable)

| Seeds fixed | Feasible codes | SAT | Lowest conditional logical error |
|---|---:|---|---:|
| None | **768** | SAT | **0.038588956%** |
| \(ZZII\) | **8** | SAT | **0.038590422%** |
| \(ZZII\) and \(IIZZ\) | **0** | UNSAT | Not defined |

The optimum **unseeded** code is

\[
S_{\varepsilon}=\langle XXII,\;YZXX,\;IIZY\rangle,
\quad [[4,1,2]].
\]

The optimum **one-seed** code is

\[
S_{H+\varepsilon}=\langle IIXX,\;ZZII,\;XXYZ\rangle,
\quad [[4,1,2]].
\]

**Conclusion:** imposing the one Hamiltonian symmetry changes the chosen code but **does not improve its modeled error rate**. Two seeds are incompatible with this correction task at this size.

## 5. Result B — probability-prioritized codes (distance ≥2, without mandatory single-Z correction)

The softer search can choose which physical errors matter most, using the same 256 probability weights. It does **not** promise to correct all single-qubit \(Z\) errors.

| Seeds fixed | Feasible codes | SAT | Lowest conditional logical error |
|---|---:|---|---:|
| None | 2,268 | SAT | **0.037866637%** |
| \(ZZII\) | 36 | SAT | **0.037868033%** |
| Both | 4 | SAT | **0.064449789%** |

The probability-prioritized search does slightly better without the strict correction guarantee. Requiring **both** Hamiltonian seeds worsens the optimal model-predicted error by approximately 70% relative to the unseeded result. This comparison is valid *within the probability-prioritized mode*; it cannot be treated as a comparison between identical correction guarantees across modes.

In [ ]:
keys = [k for k,_ in conditions]; labels = [t for _,t in conditions]
strict = [results[k]['best']['logical_failure_conditional_on_computational_survival']*100 if results[k]['best'] else np.nan for k in keys]
weighted = [results['weighted_only_d2'][k]['logical_failure_conditional_on_computational_survival']*100 for k in keys]
x = np.arange(3)
fig, ax = plt.subplots(figsize=(9,4.5))
ax.bar(x-0.18, weighted, width=0.36, label='Probability-prioritized, d≥2')
ax.bar(x+0.18, strict, width=0.36, label='All single-Z correctable, d≥2')
ax.text(x[2]+0.18,0.003,'UNSAT',ha='center')
ax.set_xticks(x,labels)
ax.set_ylabel('Logical error given computational survival (%)')
ax.set_title('Hamiltonian seed comparison, full 256-error model')
ax.legend();ax.grid(axis='y',alpha=.2)
plt.tight_layout(); plt.show()

## 6. What does this result mean for the paper?

1. We have a **working, fully checked SAT/SMT formulation** for general Pauli stabilizers at \([[4,1]]\), and its exact small-system optimum.
2. The Rydberg-derived probabilities have an **actual role**: they rank codes by decoder success, rather than assigning arbitrary priorities or selecting only the three largest errors.
3. The **Hamiltonian seed is a constraint**, not a guarantee of better error correction. With one seed there is practically no performance change; with two it can eliminate every strict solution or substantially worsen the softer one.
4. The **distance constraint matters**. An earlier 0.0341% result came from a code with **distance 1** and must **not** replace the present \([[4,1,2]]\) results.

**The limits:** The inferred four-qubit noise assumes two independent pairs. The comparison ignores residual Rydberg and loss outcomes for decoding (but reports their probability separately), and uses a Pauli approximation rather than the full Kraus map. It does not model syndrome extraction or check whether all stabilizer measurements are physically local. The added stabilizers do not necessarily commute with \(H\). No hardware-unaware comparator, logical-error-vs-noise-strength plot, or pseudothreshold has yet been produced. This is a **finite-size, noise-informed code-discovery result**, not evidence of a general hardware advantage.

**Sources:** Evered et al., *Nature* 622, 268–272 (2023), https://doi.org/10.1038/s41586-023-06481-y; Bloqade 3-level Hamiltonian https://queracomputing.github.io/Bloqade.jl/dev/3-level/; DalFavero et al., *Satisfying Quantum Codes*, arXiv:2608.23460 (2026).

In [ ]:
from pathlib import Path
from importlib.util import spec_from_file_location, module_from_spec
source=base/'rydberg_sat_study.py'
print('Solver:',source)
print('Input distribution:',base/'rydberg_pair_pauli_weights.json')
print('SAT formulas:',sorted(x.name for x in (base/'SAT_Rydberg').glob('*.smt2')))
print('To reproduce everything: python rydberg_sat_study.py')
print('To test: python -m unittest -v test_rydberg_sat_study.py')

### Repository context — Special curriculum paper

The calculation is the same in both repositories; the datasets and solver are deliberately mirrored for reproducibility. The thesis can later extend this to full leakage-aware recovery, code comparison and pseudothreshold experiments. The paper should present these as **preliminary finite-size numerical results**, not a completed hardware benchmark.